# 파라미터 스윕 (Parameter Sweep)

이 노트북에서는 복합 양자 시스템의 파라미터 의존성을 분석합니다.

**다루는 내용:**
- 단일 큐비트 자속 스윕 (`ParameterSweep`)
- 복합 시스템의 dressed 스펙트럼 스윕 (`HilbertSpaceSweep`)
- Bare vs Dressed 전이 비교
- 상태 추적 (bare/dressed 매핑)
- 분산 시프트 $\chi$ vs 자속
- 다중 큐비트 시스템

In [ ]:
using ScQubitsMimic
using CairoMakie

_scqubitsmimic_example_dir = normpath(joinpath(dirname(pathof(ScQubitsMimic)), "..", "examples"))
if !isdefined(Main, :ScQubitsMimicExampleMakie)
    include(joinpath(_scqubitsmimic_example_dir, "makie_fontsetup.jl"))
end
ScQubitsMimicExampleMakie.setup_makie_font!()


## 1. 시스템 정의

자속 가변 Transmon 큐비트와 공진기로 구성된 circuit QED 시스템을 정의합니다.

$$H = H_{\text{qubit}}(\Phi_{\text{ext}}) + H_{\text{res}} + g \, \hat{n} \otimes (\hat{a} + \hat{a}^\dagger)$$

외부 자속 $\Phi_{\text{ext}}$를 스윕하면 큐비트 주파수가 변하면서 공진기와의 상호작용이 달라집니다.

In [ ]:
tmon = TunableTransmon(EJmax=20.0, EC=0.3, d=0.1, flux=0.0, ng=0.0,
                       ncut=15, truncated_dim=4)
osc = Oscillator(E_osc=5.5, truncated_dim=6)

hs = HilbertSpace([tmon, osc])

# 결합: g * n̂_tmon ⊗ (a + a†)
g = 0.15
add_interaction!(hs, g, [tmon, osc],
    [s -> n_operator(s),
     s -> annihilation_operator(s) + creation_operator(s)])

println("시스템 구성:")
println("  TunableTransmon: EJmax=$(tmon.EJmax), EC=$(tmon.EC), d=$(tmon.d)")
println("  Oscillator: ω_r=$(osc.E_osc) GHz")
println("  결합: g=$g GHz")
println("  전체 힐베르트 공간: $(hilbertdim(hs))")

## 2. 단일 큐비트 자속 스윕

먼저 bare 큐비트(공진기 결합 없이)의 에너지 스펙트럼을 자속 함수로 계산합니다.

In [ ]:
tmon.flux = 0.0
bare_sweep = ParameterSweep(tmon, :flux, range(0.0, 0.5, length=101); evals_count=4)
plot_evals_vs_paramvals(bare_sweep; subtract_ground=true, evals_count=4)

## 3. HilbertSpace Dressed 스펙트럼 스윕

`HilbertSpaceSweep`는 복합 시스템 전체의 dressed 에너지를 파라미터 함수로 계산합니다. 자속 변화에 따라 큐비트 주파수가 공진기 주파수에 접근하면 **avoided crossing** (반교차)이 나타납니다.

In [ ]:
flux_vals = collect(range(0.0, 0.5, length=101))

sweep = HilbertSpaceSweep(hs,
    Dict(:flux => flux_vals),
    (hs, vals) -> begin
        tmon.flux = vals[:flux]
    end;
    evals_count=8
)

plot_evals_vs_paramvals(sweep; subtract_ground=true, evals_count=8)

## 4. Bare vs Dressed 전이 비교

Bare 큐비트의 전이 주파수와 dressed 전이 주파수를 비교하면 결합에 의한 에너지 이동(Lamb shift)을 확인할 수 있습니다.

In [ ]:
# Bare vs Dressed 비교 (샘플 포인트)
sample_flux = range(0.0, 0.5, length=11)
println("  Φ/Φ₀    Bare ω₀₁    Res ω     Dressed ω₀₁   Lamb shift (MHz)")
println("  " * "-"^65)
for flux in sample_flux
    tmon.flux = flux
    bare_e = eigenvals(tmon; evals_count=2)
    bare_w01 = bare_e[2] - bare_e[1]

    # swept dressed 에너지에서 가장 가까운 인덱스 찾기
    idx = argmin(abs.(flux_vals .- flux))
    dressed_w01 = sweep.dressed_evals[idx, 2] - sweep.dressed_evals[idx, 1]
    lamb = (dressed_w01 - bare_w01) * 1000

    println("  $(round(flux, digits=2))      $(round(bare_w01, digits=4))    $(osc.E_osc)    $(round(dressed_w01, digits=4))       $(round(lamb, digits=2))")
end
tmon.flux = 0.0

In [ ]:
# Bare vs Dressed 오버레이 플롯
bare_w01_arr = Float64[]
for flux in flux_vals
    tmon.flux = flux
    e = eigenvals(tmon; evals_count=2)
    push!(bare_w01_arr, e[2] - e[1])
end
tmon.flux = 0.0

dressed_w01_arr = sweep.dressed_evals[:, 2] .- sweep.dressed_evals[:, 1]

fig = Figure(size=(600, 400))
ax = Axis(fig[1, 1],
    xlabel="Φ_ext / Φ₀",
    ylabel="ω₀₁ (GHz)",
    title="Bare vs Dressed 전이 주파수")
lines!(ax, flux_vals, bare_w01_arr, label="Bare 큐비트", linewidth=2, linestyle=:dash)
lines!(ax, flux_vals, dressed_w01_arr, label="Dressed", linewidth=2)
hlines!(ax, [osc.E_osc], color=:gray, linestyle=:dot, label="공진기 ω_r")
axislegend(ax)
fig

## 5. 상태 추적 (State Tracking)

`store_lookups=true` 옵션으로 스윕하면 각 파라미터 지점에서 bare/dressed 상태 매핑을 저장합니다. 이를 통해 스윕 과정에서 에너지 준위의 "정체"를 추적할 수 있습니다.

In [ ]:
flux_fine = collect(range(0.0, 0.5, length=11))

sweep2 = HilbertSpaceSweep(hs,
    Dict(:flux => flux_fine),
    (hs, vals) -> begin
        tmon.flux = vals[:flux]
    end;
    evals_count=10,
    store_lookups=true
)

println("자속 스윕에 따른 상태 추적:")
for (i, flux) in enumerate(flux_fine)
    println("\n  Φ/Φ₀ = $(round(flux, digits=3)):")
    lk = sweep2.lookups[i]
    for di in 1:min(6, length(lk.dressed_evals))
        bi = lk.dressed_to_bare[di]
        e = lk.dressed_evals[di]
        println("    dressed[$di] → |q=$(bi[1]-1), n=$(bi[2]-1)⟩  E=$(round(e, digits=4)) GHz")
    end
end

## 6. 분산 시프트 $\chi$ vs 자속

분산 시프트는 큐비트 읽기(readout)의 핵심 파라미터입니다:

$$\chi_{ij} = E(1_i, 1_j) - E(1_i, 0_j) - E(0_i, 1_j) + E(0_i, 0_j)$$

자속에 따라 큐비트-공진기 디튜닝이 변하면서 $\chi$도 변합니다. 공진에 가까울수록 $\chi$가 커지지만, Purcell 감쇠도 증가합니다.

In [ ]:
plot_chi_vs_paramvals(sweep2; subsys_pair=(1, 2))

In [ ]:
# 수치 확인
println("분산 시프트 χ vs 자속:")
for (i, flux) in enumerate(flux_fine)
    lk = sweep2.lookups[i]
    try
        E00 = lk.dressed_evals[lk.bare_to_dressed[(1,1)]]
        E10 = lk.dressed_evals[lk.bare_to_dressed[(2,1)]]
        E01 = lk.dressed_evals[lk.bare_to_dressed[(1,2)]]
        E11 = lk.dressed_evals[lk.bare_to_dressed[(2,2)]]
        χ = (E11 - E10) - (E01 - E00)
        println("  Φ/Φ₀ = $(round(flux, digits=3)) → χ/2π = $(round(χ * 1000, digits=2)) MHz")
    catch ex
        println("  Φ/Φ₀ = $(round(flux, digits=3)) → 계산 불가")
    end
end

## 7. 다중 큐비트 시스템

두 개의 TunableTransmon 큐비트를 공진기 버스(bus resonator)를 통해 결합하는 더 복잡한 시스템을 분석합니다.

$$H = H_{q1}(\Phi_1) + H_{q2} + H_{\text{bus}} + g_1 \hat{n}_1 (\hat{a} + \hat{a}^\dagger) + g_2 \hat{n}_2 (\hat{a} + \hat{a}^\dagger)$$

In [ ]:
tmon1 = TunableTransmon(EJmax=25.0, EC=0.4, d=0.05, flux=0.0, ncut=10, truncated_dim=3)
tmon2 = TunableTransmon(EJmax=22.0, EC=0.35, d=0.08, flux=0.0, ncut=10, truncated_dim=3)
bus = Oscillator(E_osc=6.0, truncated_dim=4)

hs3 = HilbertSpace([tmon1, tmon2, bus])

g1, g2 = 0.12, 0.10
add_interaction!(hs3, g1, [tmon1, bus],
    [s -> n_operator(s),
     s -> annihilation_operator(s) + creation_operator(s)])
add_interaction!(hs3, g2, [tmon2, bus],
    [s -> n_operator(s),
     s -> annihilation_operator(s) + creation_operator(s)])

println("큐비트 1: EJmax=$(tmon1.EJmax), EC=$(tmon1.EC)")
println("큐비트 2: EJmax=$(tmon2.EJmax), EC=$(tmon2.EC)")
println("버스 공진기: ω=$(bus.E_osc) GHz")
println("전체 힐베르트 공간: $(hilbertdim(hs3))")

In [ ]:
# 큐비트 1의 자속 스윕
sweep3 = HilbertSpaceSweep(hs3,
    Dict(:flux1 => collect(range(0.0, 0.4, length=5))),
    (hs, vals) -> begin
        tmon1.flux = vals[:flux1]
    end;
    evals_count=8,
    store_lookups=true
)

println("Dressed 스펙트럼 vs 큐비트 1 자속:")
for (i, flux) in enumerate(sweep3.param_vals[:flux1])
    lk = sweep3.lookups[i]
    println("\n  Φ₁/Φ₀ = $(round(flux, digits=2)):")
    for di in 1:min(5, length(lk.dressed_evals))
        bi = lk.dressed_to_bare[di]
        e = lk.dressed_evals[di]
        println("    dressed[$di] → |$(bi[1]-1),$(bi[2]-1),$(bi[3]-1)⟩  E=$(round(e, digits=3)) GHz")
    end
end